# Lab Assignment 9 — Vision Based Cognitive Systems

**Jiya Mahajan | Roll No. 1024170296 | Batch 3Q31 / 3Q3A**


## Q1


### Objective and workflow

Build a visual assistant that reads medicine packaging and helps a user locate printed label information. The system uses LandingAI’s visual inspection workflow to locate and read text, then applies structured extraction and safety rules. It should present extracted label content as transcription, never as a treatment recommendation.

### Dataset and annotation plan

Collect clear, permissioned photographs of medicine boxes and bottles from public product-label datasets or self-photographed packaging. Include varied lighting, angles, fonts, glare, and partial occlusion. Annotate image-level fields: medicine name, strength, dosage form, active ingredient, expiry date, batch number, warnings, and manufacturer. Split by product identity into train, validation, and test sets to prevent near-duplicate leakage.

### LandingAI model design

Create a LandingAI visual inspection project. Label text regions and the field class for each region. Train an object detection model for label regions and use OCR on each detected crop; where the selected LandingAI project supports text extraction directly, map its returned text regions into the same field schema. Evaluate field-level exact match and character error rate, and report performance separately for expiry dates and warnings.

In [ ]:
from dataclasses import dataclass
from datetime import date
import re

@dataclass
class MedicineLabel:
    medicine_name: str | None = None
    strength: str | None = None
    dosage_form: str | None = None
    active_ingredient: str | None = None
    expiry_date: str | None = None
    batch_number: str | None = None
    warnings: str | None = None
    manufacturer: str | None = None

FIELD_PATTERNS = {
    "strength": r"\b\d+(?:\.\d+)?\s?(?:mg|mcg|g|ml|%)\b",
    "expiry_date": r"\b(?:EXP|EXPIRY|USE BEFORE)\s*[:.]?\s*([A-Z0-9/-]{4,12})",
    "batch_number": r"\b(?:BATCH|B\.NO|LOT)\s*[:.]?\s*([A-Z0-9-]+)",
}

def parse_ocr_text(text):
    fields = MedicineLabel()
    for attribute, pattern in FIELD_PATTERNS.items():
        match = re.search(pattern, text, flags=re.I)
        if match:
            setattr(fields, attribute, match.group(1) if match.lastindex else match.group(0))
    warning_match = re.search(r"(WARNING[^\n]*|CAUTION[^\n]*)", text, flags=re.I)
    if warning_match:
        fields.warnings = warning_match.group(0)
    return fields

example_ocr = "ACME Paracetamol 500 mg Tablets\nBATCH: A19X2\nEXP: 08/2027\nWARNING: Keep out of reach of children"
parse_ocr_text(example_ocr)

### User interaction and safety rules

1. Capture or upload an image and check that it is sharp enough to read.
2. Run detection and OCR; show the source crop next to each extracted field.
3. Let the user edit uncertain text and flag low-confidence or missing fields for manual confirmation.
4. Display expiry and warnings exactly as printed, with a reminder to confirm important details with a pharmacist.
5. Do not infer a dose, diagnose a condition, or recommend medicine based on an image.
6. Delete uploaded images according to the project’s privacy policy; do not use identifiable user data for training without consent.

### Evaluation and deployment

Measure detection precision/recall, OCR character error rate, and exact-match accuracy for each field. Test glare, blur, rotated labels, multiple languages, and similar packaging. Deploy a small upload interface that shows the image, extracted fields, confidence, and correction controls. The notebook includes the extraction schema and post-processing logic; connecting a deployed LandingAI endpoint requires its project endpoint and API credentials.

## Q2


### Objective and data

Use a facial-expression dataset with consented, diverse examples and labels such as happy, sad, angry, fearful, surprised, disgusted, and neutral. Separate training and test subjects to reduce identity leakage. Facial-expression classifiers estimate visible expression; they do not establish a person’s actual emotional state.

### Pipeline and assistant behavior

Detect a face, align and normalize the crop, run a CNN expression classifier, then apply a confidence threshold. Ask the user whether the suggested interpretation is right before adapting the conversation. If confidence is low, respond neutrally and ask how the user feels. Never use the prediction for high-impact decisions or to infer sensitive traits.

In [ ]:
def assistant_response(label, confidence, threshold=0.60):
    if confidence < threshold:
        return "I may not be reading your expression correctly. How are you feeling?"
    prompts = {
        "happy": "You seem upbeat. What would you like to work on?",
        "sad": "Would you like to talk about what is on your mind?",
        "angry": "Would a short pause or a different kind of help be useful?",
        "fearful": "Would you like me to explain the next step more slowly?",
        "surprised": "That looks unexpected. Would you like more context?",
        "disgusted": "Would you prefer to change the topic or approach?",
        "neutral": "How can I help you today?",
    }
    return prompts.get(label, "How can I help you today?")

print(assistant_response("sad", 0.82))
print(assistant_response("angry", 0.31))

### Evaluation, privacy, and limitations

Report macro F1, per-class recall, calibration, and subgroup performance across lighting, skin tone, age, and camera conditions. Obtain explicit consent, process frames locally where possible, avoid storing face images, and provide an opt-out. Expressions vary by culture and individual, so the assistant asks rather than assumes and gives the user control over the interpretation.